# ARGUS RFBL enrollment embeddings (T4 GPU)

Enrolls the organizer-provided RFBL dataset (`datasets/RFBL/RFBL_Register/`, 460 identities,
one photo each, ~100-160px) into the same ChromaDB gallery schema as the LFW/MFR2 demo
gallery: one `UNMASKED` template per identity, plus synthetic masked templates
(surgical, surgical_blue, N95, KN95, cloth - the same 5 mask types the local pipeline
settled on) for masking-robust matching.

Runs entirely on Colab because it needs a GPU for the embedding step to be fast, but the
masking step (MaskTheFace, CPU/dlib-bound) also runs here so the whole pipeline is one run.

**Before running:**
1. Locally: `python -m enrollment.build_rfbl_manifest` then `python -m enrollment.package_rfbl_for_colab`
   - this produces `enrollment/rfbl_colab_bundle.zip` (~22MB: 460 images + the vendored
     MaskTheFace tool, minus its 96MB dlib model which this notebook downloads itself).
2. Upload `rfbl_colab_bundle.zip` to a folder in your Google Drive, e.g. `MyDrive/argus_enrollment/`.
   Update `DRIVE_DIR` in the next cell if you used a different folder name.
3. **Runtime -> Change runtime type -> T4 GPU**, before running any cell.
4. Run cells top to bottom. If Colab prompts a restart after the install cell (needed so the
   freshly installed GPU onnxruntime is what actually gets imported, not a CPU build pulled
   in transitively by another package), restart, then just re-run the provider-check cell
   below before continuing - installs persist across a session restart, nothing else needs
   to be redone.

## 1. Install dependencies

`onnxruntime` (CPU) and `onnxruntime-gpu` share the same import name and clobber each other
if both are installed - this bit the local enrollment pipeline once already (see
`enrollment/README.md`). Uninstalling the CPU package first, then installing only the GPU
one, avoids that. `dlib` has no prebuilt wheel for every Colab Python version, so `cmake` is
installed defensively in case it needs to build from source (a couple of minutes, not
instant).

In [ ]:
!apt-get -qq install -y cmake > /dev/null
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu insightface opencv-python-headless dlib imutils face-recognition face-recognition-models dotmap Pillow requests

print('install done - if Colab shows a "Restart session" prompt, click it, then continue from the next cell')

In [ ]:
import onnxruntime

providers = onnxruntime.get_available_providers()
print('available providers:', providers)
assert 'CUDAExecutionProvider' in providers, (
    'No CUDA provider found - check Runtime > Change runtime type > T4 GPU, then Runtime > Restart session'
)

## 2. Mount Drive and extract the bundle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/argus_enrollment'
BUNDLE_ZIP = f'{DRIVE_DIR}/rfbl_colab_bundle.zip'

Unzips into `images/` (460 originals, renamed to `{identity}.ext`) and `masktheface/`
(the vendored tool + mask assets, no dlib model).

In [ ]:
import zipfile
import os

EXTRACT_DIR = '/content/rfbl_bundle'
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

IMAGES_DIR = os.path.join(EXTRACT_DIR, 'images')
MASKTHEFACE_DIR = os.path.join(EXTRACT_DIR, 'masktheface')
MASKED_DIR = IMAGES_DIR + '_masked'

print(len(os.listdir(IMAGES_DIR)), 'original images extracted')

## 3. Run MaskTheFace

Same invocation as the local pipeline's `datasets/masking/scripts/run_masktheface.py`,
including the `--color ""` fix: MaskTheFace's own default (`#0473e2`) tints every mask the
same blue regardless of its native color unless explicitly overridden. `mask_type all`
generates 9 variants per image; the cell after this one keeps only the 5 the local pipeline
already settled on (`KEEP_MASK_TYPES` below), for consistency with the rest of the gallery.

The dlib landmark model downloads automatically on first use (`dlib.net`, ~96MB) - this is
the slow part of this cell (one-time), not the masking itself.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'mask_the_face.py',
    '--path', IMAGES_DIR,
    '--mask_type', 'all',
    '--color', '',
]
subprocess.run(cmd, cwd=MASKTHEFACE_DIR, check=True)

masked_files = os.listdir(MASKED_DIR)
print(len(masked_files), 'masked files written to', MASKED_DIR)

## 4. Build the embedding worklist

Parses `{identity}_{mask_type}.ext` back into `(identity, mask_type)` - safe because no RFBL
identity name contains an underscore (verified locally before packaging). Filters down to
the 5 kept mask types, and adds the 460 originals as `mask_type=UNMASKED`.

In [ ]:
KEEP_MASK_TYPES = {'surgical', 'surgical_blue', 'N95', 'KN95', 'cloth'}

worklist = []

for filename in sorted(os.listdir(IMAGES_DIR)):
    identity = os.path.splitext(filename)[0]
    worklist.append({
        'identity': identity,
        'mask_type': 'UNMASKED',
        'is_masked': 0,
        'path': os.path.join(IMAGES_DIR, filename),
    })

for filename in sorted(masked_files):
    stem = os.path.splitext(filename)[0]
    identity, mask_type = stem.split('_', 1)
    if mask_type not in KEEP_MASK_TYPES:
        continue
    worklist.append({
        'identity': identity,
        'mask_type': mask_type,
        'is_masked': 1,
        'path': os.path.join(MASKED_DIR, filename),
    })

print(f'{len(worklist)} images to embed '
      f'({sum(1 for w in worklist if w["is_masked"] == 0)} unmasked, '
      f'{sum(1 for w in worklist if w["is_masked"] == 1)} masked)')

## 5. Load ArcFace (buffalo_l) on the T4

`det_size=(160, 160)`, not InsightFace's default `640x640` - RFBL's originals are
~100-160px, and the local pipeline already found that upsampling small pre-cropped images
into a 640x640 canvas breaks SCRFD's anchor matching almost entirely (this is exactly the
RWMFD/MFR2 failure documented in `embeddings/generate.py` and `enrollment/README.md`;
RFBL's images are in the same small pre-cropped regime, so the same fix applies here).

In [ ]:
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
app.prepare(ctx_id=0, det_size=(160, 160))

## 6. Generate embeddings

In [ ]:
import cv2
import numpy as np

def pick_largest_face(faces):
    if not faces:
        return None
    return max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

kept, embeddings = [], []
skipped = 0

for i, item in enumerate(worklist):
    image = cv2.imread(item['path'])
    if image is None:
        skipped += 1
        continue
    face = pick_largest_face(app.get(image))
    if face is None:
        skipped += 1
        continue
    kept.append(item)
    embeddings.append(face.normed_embedding.astype(np.float32))
    if (i + 1) % 200 == 0:
        print(f'{i + 1}/{len(worklist)} processed, {skipped} skipped')

print(f'done: {len(kept)} embeddings, {skipped} skipped')

## 7. Save to Drive

Same column layout `enrollment/seed_rfbl.py` (run locally after downloading this file)
expects: `dataset`, `identity`, `mask_type`, `is_masked`, `path`, `embedding`.

In [ ]:
out_path = os.path.join(DRIVE_DIR, 'rfbl_embeddings.npz')
np.savez_compressed(
    out_path,
    dataset=np.array(['rfbl'] * len(kept)),
    identity=np.array([item['identity'] for item in kept]),
    mask_type=np.array([item['mask_type'] for item in kept]),
    is_masked=np.array([item['is_masked'] for item in kept]),
    path=np.array([item['path'] for item in kept]),
    embedding=np.stack(embeddings),
)
print('saved to', out_path)

`rfbl_embeddings.npz` is now in your Drive folder. Download it into `enrollment/` locally
(or sync the Drive folder) and run `python -m enrollment.seed_rfbl` to upsert these templates
into `backend/.chroma`, the same way `enrollment/seed_chroma.py` seeded the LFW/MFR2 gallery.